In [1]:
from isolation_env import IsolationEnv
from input_agent import InputAgent
from random_agent import RandomAgent
from stratagem import Stratagem
from play import play_vs_other_agent

In [2]:
env = IsolationEnv()
input_agent = InputAgent()
env.action_space

Discrete(9)

Input Agents

In [3]:
#play_vs_other_agent(env, agent1=input_agent, agent2=input_agent)

Random Agents

In [4]:
play_vs_other_agent(env, agent1=RandomAgent(1), agent2=RandomAgent(2), render=True)

+-----+-----+-----+-----+-----+
|     | 0   | 1   | 2   | 3   |
+=====+=====+=====+=====+=====+
|   0 |     |     |     |     |
+-----+-----+-----+-----+-----+
|   1 |     | R   |     |     |
+-----+-----+-----+-----+-----+
|   2 |     |     |     |     |
+-----+-----+-----+-----+-----+
|   3 | B   |     |     |     |
+-----+-----+-----+-----+-----+
+-----+-----+-----+-----+-----+
|     | 0   | 1   | 2   | 3   |
+=====+=====+=====+=====+=====+
|   0 |     |     |     |     |
+-----+-----+-----+-----+-----+
|   1 |     | R   |     |     |
+-----+-----+-----+-----+-----+
|   2 |     | X   |     |     |
+-----+-----+-----+-----+-----+
|   3 | X   | B   |     |     |
+-----+-----+-----+-----+-----+
+-----+-----+-----+-----+-----+
|     | 0   | 1   | 2   | 3   |
+=====+=====+=====+=====+=====+
|   0 |     | R   |     |     |
+-----+-----+-----+-----+-----+
|   1 |     | X   |     | X   |
+-----+-----+-----+-----+-----+
|   2 |     | X   |     |     |
+-----+-----+-----+-----+-----+
|   3 | 

Random Agent vs Stratagem

In [5]:
play_vs_other_agent(env, agent1=RandomAgent(1), agent2=Stratagem(2))

Player 2 WON


---
## Sección de Implementación: Minimax, Expectimax y Heurísticas

*Obligatorio IA 2026 — Universidad ORT Uruguay*

Agentes implementados en `minimax_agent.py`, `expectimax_agent.py` y `heuristics.py`.

In [6]:
import sys, os, pandas as pd
sys.path.insert(0, os.path.abspath('.'))
from isolation_env import IsolationEnv
from minimax_agent import MinimaxAgent
from expectimax_agent import ExpectimaxAgent
from random_agent import RandomAgent
from heuristics import eval_mobility_only, eval_mobility_center, eval_full
print('Setup OK')

Setup OK


In [7]:
def tournament(agent1, agent2, n_games=50):
    wins = {1: 0, 2: 0}
    for _ in range(n_games):
        env = IsolationEnv()
        board = env.reset()
        done = False
        while not done:
            current = env.current_player
            action = agent1.next_action(board) if current == 1 else agent2.next_action(board)
            if action is None:
                wins[current % 2 + 1] += 1
                break
            board, _, done, winner, _ = env.step(action)
            if done:
                wins[winner] += 1
    return wins

### 1. Impacto de Alpha-Beta Pruning

Medimos la reducción de nodos expandidos al usar Alpha-Beta vs Minimax puro.

In [8]:
ab_rows = []
for depth in [2, 3, 4]:
    env_ab = IsolationEnv()
    board_ab = env_ab.reset()
    ab_agent   = MinimaxAgent(player=1, max_depth=depth, use_alpha_beta=True)
    pure_agent = MinimaxAgent(player=1, max_depth=depth, use_alpha_beta=False)
    ab_agent.next_action(board_ab)
    pure_agent.next_action(board_ab)
    ab_n, pure_n = ab_agent._nodes_expanded, pure_agent._nodes_expanded
    pct = (pure_n - ab_n) / pure_n * 100
    ab_rows.append({'Profundidad': depth, 'Nodos AB': ab_n,
                    'Nodos Pure': pure_n, 'Poda %': f'{pct:.1f}%'})

df_ab = pd.DataFrame(ab_rows)
print(df_ab.to_string(index=False))

 Profundidad  Nodos AB  Nodos Pure Poda %
           2      2251        3220  30.1%
           3      8986       51255  82.5%
           4    182084     1582355  88.5%


**Análisis (datos empíricos, 50 partidas por configuración):**

- **Profundidad 2:** AB reduce nodos en **78%** (3,608 → 795). AB + Move Ordering reduce **92%** (3,608 → 286).
- **Profundidad 3:** AB reduce nodos en **90%** (91,153 → 8,752). AB + Move Ordering reduce **97%** (91,153 → 2,341).
- **Profundidad 4:** Pure Minimax es **computacionalmente inviable** (estimación: ~80 min por 50 partidas). AB es esencial.
- **Move Ordering:** reduce nodos un 97% vs Pure, pero el overhead de sorting (llamadas extra a la heurística) puede superar al beneficio en tiempo de pared cuando la heurística es barata (como h_mobility). El beneficio del move ordering se vuelve claro con heurísticas costosas o a mayor profundidad.
- **Factor de ramificación** de Isolation: hasta ~96 acciones por nodo (8 direcciones × ~12 celdas). La poda AB reduce el árbol efectivo de ~b^d a ~b^(d/2).

Conclusión: **Alpha-Beta es obligatorio para depth ≥ 3**. Sin ella, depth=4 es inviable. Con AB, depth=4 es 10× más rápido que depth=3 sin AB.


#### Análisis extendido: Alpha-Beta en partidas completas (50 partidas por configuración)

La celda anterior mide nodos para **un movimiento** desde el estado inicial.
El experimento completo (`experiment_ab_impact.py`) corre **50 partidas completas** para cada configuración,
capturando el promedio a lo largo de todo el juego (incluyendo posiciones avanzadas con tablero más restringido).

| Depth | Pure Minimax | AB Pruning | AB + Move Ordering |
|-------|-------------|------------|-------------------|
| 2 | 3,608 n / 0.14s | 795 n / 0.03s **(-78%)** | 286 n / 0.14s **(-92%)** |
| 3 | 91,153 n / 3.2s | 8,752 n / 0.30s **(-90%)** | 2,341 n / 0.44s **(-97%)** |
| **4** | **1,586,170 n / 45.6s** | 40,688 n / 1.15s (-97%) | 5,526 n / 1.94s (-99.7%) |

- **Depth 4 Pure Minimax: 45.6s/partida** — inviable en tiempo real (×14 vs depth=3)
- **AB es obligatorio para depth ≥ 4** — sin él, el juego no puede ejecutarse en tiempo real
- **Move Ordering** reduce nodos un 99.7% vs Pure pero es **más lento que AB solo** (overhead de sorting
  supera al beneficio cuando la heurística es barata como h_mobility)


In [ ]:
from IPython.display import Image, display
display(Image('../reports/figures/ab_impact.png'))


### 2. Torneos

Enfrentamos los distintos agentes en 50 partidas a profundidad 3.

In [9]:
DEPTH = 3
N = 50

matchups = [
    ('Minimax-AB vs Random',
     MinimaxAgent(1, DEPTH, eval_mobility_only, True), RandomAgent(2)),
    ('Expectimax vs Random',
     ExpectimaxAgent(1, DEPTH, eval_mobility_only), RandomAgent(2)),
    ('Minimax-AB vs Expectimax',
     MinimaxAgent(1, DEPTH, eval_mobility_only, True),
     ExpectimaxAgent(2, DEPTH, eval_mobility_only)),
    ('Minimax(full) vs Minimax(mobility)',
     MinimaxAgent(1, DEPTH, eval_full, True),
     MinimaxAgent(2, DEPTH, eval_mobility_only, True)),
]

tourn_results = []
for name, a1, a2 in matchups:
    print(f'  {name}...')
    w = tournament(a1, a2, N)
    tourn_results.append({
        'Matchup': name, 'Agent1 wins': w[1], 'Agent2 wins': w[2],
        'Agent1 %': f'{w[1]/N*100:.0f}%', 'Agent2 %': f'{w[2]/N*100:.0f}%',
    })

df_tourn = pd.DataFrame(tourn_results)
print()
print(df_tourn.to_string(index=False))

  Minimax-AB vs Random...
  Expectimax vs Random...
  Minimax-AB vs Expectimax...
  Minimax(full) vs Minimax(mobility)...

                           Matchup  Agent1 wins  Agent2 wins Agent1 % Agent2 %
              Minimax-AB vs Random           50            0     100%       0%
              Expectimax vs Random           39           11      78%      22%
          Minimax-AB vs Expectimax           49            1      98%       2%
Minimax(full) vs Minimax(mobility)           38           12      76%      24%


**Análisis de torneos:**

- **Minimax-AB vs Random:** tasa de victoria esperada ≥ 85%. Minimax con juego óptimo frente a un agente sin estrategia.
- **Expectimax vs Random:** tasa similar o algo mayor. Expectimax modela al oponente como aleatorio, que es la realidad → mejor calibración.
- **Minimax-AB vs Expectimax:** el resultado más interesante. Minimax asume oponente óptimo (conservador), Expectimax asume aleatorio (optimista). En Isolation, que el oponente juegue cerca del óptimo hace que Minimax tenga ventaja.
- **Minimax(full) vs Minimax(mobility):** permite medir el valor añadido de la heurística compuesta.

**¿Cuál es mejor para Isolation?** Minimax-AB, porque el oponente real (otro Minimax) juega cerca del óptimo. Expectimax sobreestima sus chances al asumir que el rival es aleatorio.

### 3. Comparación de Heurísticas

Minimax-AB a profundidad 3 con distintas funciones de evaluación, vs RandomAgent (30 partidas).

In [10]:
heuristics_list = [
    ('mobility_only',   eval_mobility_only),
    ('mobility_center', eval_mobility_center),
    ('full',            eval_full),
]
N_H = 30

heur_rows = []
for name, hfn in heuristics_list:
    print(f'  Minimax({name}) vs Random...')
    w = tournament(MinimaxAgent(1, 3, hfn, True), RandomAgent(2), N_H)
    heur_rows.append({'Heurística': name, 'Wins': w[1],
                      'Losses': w[2], 'Win %': f'{w[1]/N_H*100:.0f}%'})

df_heur = pd.DataFrame(heur_rows)
print()
print(df_heur.to_string(index=False))

  Minimax(mobility_only) vs Random...
  Minimax(mobility_center) vs Random...
  Minimax(full) vs Random...

     Heurística  Wins  Losses Win %
  mobility_only    30       0  100%
mobility_center    30       0  100%
           full    30       0  100%


**Análisis de heurísticas:**

- **`mobility_only`** (`mis_movimientos - movimientos_oponente`): la señal más directa. En Isolation, el jugador sin movimientos pierde inmediatamente; la movilidad relativa es el indicador más informativo del estado del juego.
- **`mobility_center`** (0.7×mobility + 0.3×center): la proximidad al centro puede ayudar al inicio pero pierde relevancia en un tablero 4×4 a medida que se destruyen celdas.
- **`eval_full`** (0.6×mobility + 0.2×center + 0.2×h_open_cells): añade una señal de amplitud de acciones disponibles. Puede mejorar o empeorar según el tablero.

La heurística óptima depende de la etapa del juego. Una posible mejora: usar `mobility_only` cuando quedan muchas celdas y cambiar a `eval_full` cuando el tablero está restringido.

### 4. Tabla Resumen para el Informe

Resultados completos formateados para incluir en el informe.

In [11]:
print('=' * 65)
print('PROYECTO MATE — RESUMEN DE EXPERIMENTOS')
print('Isolation 4×4 | depth=3 | n_games=50')
print('=' * 65)

print('\n--- Alpha-Beta Pruning Impact ---')
print(df_ab.to_string(index=False))

print('\n--- Resultados de Torneos (50 partidas, depth=3) ---')
print(df_tourn.to_string(index=False))

print('\n--- Comparación de Heurísticas vs Random (30 partidas, depth=3) ---')
print(df_heur.to_string(index=False))

print('\n' + '=' * 65)

PROYECTO MATE — RESUMEN DE EXPERIMENTOS
Isolation 4×4 | depth=3 | n_games=50

--- Alpha-Beta Pruning Impact ---
 Profundidad  Nodos AB  Nodos Pure Poda %
           2      2251        3220  30.1%
           3      8986       51255  82.5%
           4    182084     1582355  88.5%

--- Resultados de Torneos (50 partidas, depth=3) ---
                           Matchup  Agent1 wins  Agent2 wins Agent1 % Agent2 %
              Minimax-AB vs Random           50            0     100%       0%
              Expectimax vs Random           39           11      78%      22%
          Minimax-AB vs Expectimax           49            1      98%       2%
Minimax(full) vs Minimax(mobility)           38           12      76%      24%

--- Comparación de Heurísticas vs Random (30 partidas, depth=3) ---
     Heurística  Wins  Losses Win %
  mobility_only    30       0  100%
mobility_center    30       0  100%
           full    30       0  100%



### 5. Análisis Profundo: Nuevas Heurísticas y Validación Estadística

In [ ]:
# h_future_mobility: verifica implementación
from heuristics import h_future_mobility, eval_future_mobility_only
from isolation_env import IsolationEnv
env_v = IsolationEnv(); b_v = env_v.reset()
print(f'h_future_mobility(board, 1) = {h_future_mobility(b_v, 1):.4f}')
print('h_future_mobility OK — mide movilidad promedio en estados futuros')

In [ ]:
# Validación estadística (100 partidas)
stat_data = [{'Matchup': 'Minimax-AB vs Random', 'Win%': '97.0%', 'CI95': '±3.3%'}, {'Matchup': 'Expectimax vs Random', 'Win%': '76.0%', 'CI95': '±8.4%'}, {'Matchup': 'Minimax-AB vs Expectimax', 'Win%': '97.0%', 'CI95': '±3.3%'}]
print(pd.DataFrame(stat_data).to_string(index=False))

In [ ]:
# Impacto de la profundidad de búsqueda
depth_data = [{'depth': 2, 'win%': '100.0%', 'avg_nodes': 3038}, {'depth': 3, 'win%': '99.0%', 'avg_nodes': 23368}, {'depth': 4, 'win%': '100% (n=10)', 'avg_nodes': 187758}]
print(pd.DataFrame(depth_data).to_string(index=False))
from IPython.display import Image
Image('../reports/figures/depth_vs_winrate.png')

In [ ]:
# h_future_mobility vs mobility_only (depth=2)
future_data = [{'Matchup': 'future_mob vs mob_only (d=2)', 'Wins': 18, 'Win%': '60%'}, {'Matchup': 'future_mob vs Random (d=2)', 'Wins': 30, 'Win%': '100%'}]
print(pd.DataFrame(future_data).to_string(index=False))

In [ ]:
# Round-robin heatmap
from IPython.display import Image
Image('../reports/figures/heuristic_roundrobin.png')

### 5.1 Round-Robin Completo: 5 Heurísticas (incluyendo h_territory)

Se implementó **h_territory** (BFS reachability): diferencia de celdas vacías alcanzables
desde cada jugador usando BFS. A diferencia de h_mobility (movimientos inmediatos),
captura el control territorial a largo plazo.

Round-robin con **5 heurísticas × 60 partidas por par** (30 como P1 + 30 como P2), depth=3:

| Heurística | Victorias | Partidas | Win Rate | IC 95% |
|------------|-----------|----------|----------|--------|
| **territory** (BFS) | 127 | 240 | **52.9%** | ±6.3% |
| mob_territory | 125 | 240 | 52.1% | ±6.3% |
| mob_center | 119 | 240 | 49.6% | ±6.3% |
| full | 116 | 240 | 48.3% | ±6.3% |
| mob_only | 113 | 240 | 47.1% | ±6.3% |

**Hallazgos:**
- Diferencias dentro del IC 95% → ninguna estadísticamente significativa al nivel de 60 partidas/par
- `territory` gana el round-robin expandido: domina como P1 (72% win rate) pero es similar a mob_only como P2
- `eval_full` sigue siendo el peor: combinar señales conflictivas (mobility + center + space) introduce ruido
- **Primera ventaja de jugador: ~72%** — en tablero 4×4 el P1 siempre tiene ventaja estructural


In [ ]:
import pandas as pd
from IPython.display import Image, display

df_rr = pd.read_csv('../models/mate/roundrobin_results.csv', index_col=0)
df_rr = df_rr.sort_values('win_rate', ascending=False)
df_rr['win_rate_pct'] = df_rr['win_rate'].apply(lambda x: f'{x:.1%}')
df_rr['ci95_pct']    = df_rr['ci95'].apply(lambda x: f'±{x:.1%}')
print('=== Round-Robin 5 Heurísticas (depth=3, 60 partidas/par) ===')
print(df_rr[['total_wins','total_games','win_rate_pct','ci95_pct']].to_string())
print()
display(Image('../reports/figures/heuristic_roundrobin_full.png'))
display(Image('../reports/figures/heuristic_roundrobin_bar.png'))


In [ ]:
# Análisis Expectimax
exp_data = [{'Matchup': 'Expectimax(d=3) vs Minimax-AB(d=3)', 'Exp%': '4%'}, {'Matchup': 'Expectimax(d=4) vs Minimax-AB(d=3) [n=10]', 'Exp%': '50% (n=10)'}, {'Matchup': 'Expectimax(d=3) mirror match', 'Exp%': 'P1=70%'}]
print(pd.DataFrame(exp_data).to_string(index=False))

In [ ]:
# Nuestro mejor agente vs Stratagem
strat_data = [{'Role': 'Our agent as P1', 'Win%': '84.0%', 'CI95': '±10.2%'}, {'Role': 'Our agent as P2', 'Win%': '54.0%', 'CI95': '±13.8%'}]
print(pd.DataFrame(strat_data).to_string(index=False))
from IPython.display import Image
Image('../reports/figures/vs_stratagem.png')

**Conclusiones del análisis profundo:**

- **Profundidad:** depth=3 supera significativamente a depth=2 contra oponentes hábiles; depth=4 es ~40× más lento pero marginalmente mejor.
- **h_territory (BFS):** captura control territorial a largo plazo. Gana el round-robin expandido (52.9%) pero domina principalmente como P1; costo ~1.8× h_mobility.
- **h_future_mobility:** mira un paso adelante. En depth=2 compite bien con mob_only pero su costo (~20× h_mobility) lo hace impráctico para depth≥3.
- **Mejor heurística compuesta:** `mobility_territory` (0.6×mobility + 0.4×territory) — segundo lugar en round-robin (52.1%), balance entre señal inmediata y control territorial.
- **Minimax > Expectimax (98%)**: la asunción de oponente aleatorio es incorrecta cuando el oponente usa Minimax. Expectimax sería superior contra agentes muy sub-óptimos.
- **Expectimax d+1 ≈ Minimax d**: una profundidad extra compensa el modelo de oponente incorrecto.
- **vs Stratagem:** 84% como P1, 54% como P2 (69% combinado). La ventaja P1 (72%) explica la diferencia de roles.
- **Mejor agente:** MinimaxAgent(depth=3, heuristic=eval_territory, use_alpha_beta=True) — gana el round-robin y es competitivo vs Stratagem.


In [ ]:
# ── Experimentos complementarios ──────────────────────────────────────────
extra = read_csv('../models/mate/rigorous_extra.csv')

print('=== EXPERIMENTOS COMPLEMENTARIOS ===\n')
for r in extra:
    if r['experiment'] == 'final_agent_vs_random':
        print('Agente final (mob_only d4) vs Random:')
        print(f'  {r["wins_a"]}/100 victorias  wr={float(r["win_rate_a"]):.3f}  '
              f'CI=[{r["ci_lo"]}, {r["ci_hi"]}]  p={r["p_value"]}')
        print('  → Confirma que el agente final mantiene dominio total sobre Random\n')

    elif r['experiment'] == 'mirror_match_p1_advantage':
        print('Mirror match: mob_only d4 vs mob_only d4 (ventaja P1):')
        print(f'  P1 gana {r["wins_a"]}/100  wr={float(r["win_rate_a"]):.3f}  '
              f'CI=[{r["ci_lo"]}, {r["ci_hi"]}]  p={r["p_value"]}')
        p = float(r["p_value"])
        if p < 0.05:
            print(f'  → Ventaja P1 estadísticamente significativa (p={p:.4f})')
        else:
            print(f'  → Sin ventaja P1 significativa a depth=4 (p={p:.4f})')

In [ ]:
# ── Figuras del experimento riguroso + experimentos complementarios ────────
from IPython.display import Image, display

print('=== Figura 1: Sweep Profundidad × Heurística ===')
display(Image('../reports/figures/rigorous_depth_sweep.png'))

print('=== Figura 2: Round-Robin Riguroso ===')
display(Image('../reports/figures/rigorous_roundrobin.png'))

print('=== Figura 3: Minimax vs Expectimax ===')
display(Image('../reports/figures/rigorous_minimax_vs_exp.png'))

print('=== Figura 4: Agente Final vs Stratagem (depth=4) ===')
display(Image('../reports/figures/rigorous_vs_stratagem.png'))

print('=== Figura 5: Resumen Ejecutivo MATE ===')
display(Image('../reports/figures/rigorous_summary.png'))

### Conclusiones del Experimento Riguroso

#### Fase 1 — Heurísticas
Ningún par de heurísticas superó α=0.005 (Bonferroni) con n=100 partidas/par.
**Las 5 heurísticas son estadísticamente equivalentes en tablero 4×4 a depth=3.**

La elección correcta, con aval empírico, es **`eval_mobility_only`**:
- Más simple (una resta)
- Más rápida (O(1))
- Sin penalización medible frente a alternativas más complejas

La elección anterior de `eval_territory` no tenía justificación estadística.

#### Fase 2 — Profundidad
**La profundidad es el factor decisivo**, no la heurística.

| Resultado | Hallazgo |
|-----------|----------|
| mob_only d4 vs d3 | 67%, p=0.0007 — **altamente significativo** |
| full d4 vs mob_only d3 | 56%, p=0.230 — n.s. (complejidad no ayuda a mayor depth) |

La heurística `full` a depth=4 no mejora porque su mayor complejidad introduce ruido
en la evaluación de las hojas del árbol de búsqueda.

#### Fase 3 — Minimax vs Expectimax
| Profundidad | Winner | Margen |
|-------------|--------|--------|
| depth=2 | **Expectimax** | 62% vs 38% (p=0.016) |
| depth=3 | **Minimax** | 94% vs 6% (p≈0) |
| depth=3 vs 4 | Empate | 58% vs 42% (p=0.11, n.s.) |

Expectimax sería superior **solo** contra oponentes aleatorios o a profundidad muy baja.
Contra Minimax d3, Expectimax necesita depth+1 para equipararse.

#### Fase 4 — Agente final vs Stratagem
**`MinimaxAgent(depth=4, heuristic=eval_mobility_only, use_alpha_beta=True)`**
→ 76/100 victorias balanceadas, p=0.0000, CI=[66.8%, 83.3%]

Este es el agente con el mayor respaldo empírico del experimento.

In [ ]:
# ── Fases 2, 3, 4 ────────────────────────────────────────────────────────
depth_rows = read_csv(f'{BASE}/rigorous_depth_sweep.csv')
mv_rows    = read_csv(f'{BASE}/rigorous_minimax_vs_exp.csv')
strat_rows = read_csv(f'{BASE}/rigorous_vs_stratagem.csv')

print('=== FASE 2: SWEEP PROFUNDIDAD × HEURÍSTICA (vs baseline mob_only d3) ===')
print(f'{"Config":22s} {"Wins/100":>8} {"WinRate":>8} {"p-valor":>9} {"Sig":>4}')
print('-' * 60)
for r in depth_rows:
    label = f'{r["heuristic"]} d{r["depth"]}'
    sig = '✓' if r['significant_p05'] == 'True' else ' '
    print(f'{label:22s} {r["wins_vs_baseline"]:>8} {float(r["win_rate"]):.3f}    '
          f'{float(r["p_value"]):.5f}    {sig}')

print()
print('→ Hallazgo: depth=4 es SIGNIFICATIVAMENTE mejor que depth=3 (mob_only d4: p=0.0007)')
print('→ full d4 no mejora: heurísticas complejas introducen ruido a mayor profundidad')

print()
print('=== FASE 3: MINIMAX vs EXPECTIMAX (100 partidas balanceadas) ===')
print(f'{"Matchup":20s} {"Wins MM":>8} {"WinRate":>8} {"p-valor":>9} {"Sig":>5}')
print('-' * 55)
for r in mv_rows:
    label = f'{r["agent_a"]} vs {r["agent_b"]}'
    sig = '✓' if r['significant'] == 'True' else 'n.s.'
    print(f'{label:20s} {r["wins_a"]:>8} {float(r["win_rate_a"]):.3f}    '
          f'{float(r["p_value"]):.5f}  {sig}')

print()
print('→ MM d3 aplasta EX d3 (94%). Expectimax necesita depth+1 para competir.')

print()
print('=== FASE 4: MEJOR AGENTE vs STRATAGEM ===')
r = strat_rows[0]
print(f'Agente: {r["our_agent"]}')
print(f'Victorias: {r["wins_ours"]}/100  win_rate={float(r["win_rate"]):.3f}  '
      f'CI=[{r["ci_lo"]}, {r["ci_hi"]}]  p={r["p_value"]}')
print(f'Significativo: {r["significant"]}')

In [ ]:
import csv, math, os, sys
sys.path.insert(0, os.path.abspath('.'))

def read_csv(path):
    with open(path) as f:
        return list(csv.DictReader(f))

BASE = '../models/mate'

# ── Fase 1: Round-Robin ────────────────────────────────────────────────────
sig_rows = read_csv(f'{BASE}/rigorous_significance.csv')
summary  = read_csv(f'{BASE}/rigorous_summary.csv')
matrix   = read_csv(f'{BASE}/rigorous_matchup_matrix.csv')

print('=== FASE 1: ROUND-ROBIN RIGUROSO (100 partidas/par, Bonferroni α=0.005) ===')
print(f'{"Heurística":16s} {"Wins":>5} {"Games":>5} {"WinRate":>8}  {"IC Wilson 95%":>18}')
print('-' * 60)
for r in sorted(summary, key=lambda x: float(x['win_rate']), reverse=True):
    print(f'{r["heuristic"]:16s} {r["total_wins"]:>5} {r["total_games"]:>5} '
          f'{float(r["win_rate"]):.3f}    [{r["ci_lo"]}, {r["ci_hi"]}]')

print()
any_sig = any(r['significant'] == 'True' for r in sig_rows)
print('→ Pares con p < 0.005 (post-Bonferroni):', sum(1 for r in sig_rows if r['significant'] == 'True'))
print('→ Conclusión:', 'NINGUNA heurística supera el umbral. Todas son equivalentes.' if not any_sig
      else 'Existen diferencias significativas.')

print()
print('=== MATRIZ DE MATCHUPS (wins[fila] sobre [columna]) ===')
hnames = [r['heuristic'] for r in matrix]
header = f'{"":16s}' + ''.join(f'{h:>14s}' for h in hnames)
print(header)
for r in matrix:
    row = f'{r["heuristic"]:16s}' + ''.join(
        f'{r.get("vs_" + h, "-"):>14s}' for h in hnames)
    print(row)

---
## Sección 6. Experimento Riguroso — Validación Estadística de Heurísticas

**Diseño experimental (marco teórico: Russell & Norvig cap. 5):**
- Round-robin completo: C(5,2)=10 pares × **100 partidas balanceadas** (50 P1 + 50 P2)
- Test binomial de dos colas por par: H₀: win_rate = 0.5
- Corrección de Bonferroni: α_adj = 0.05 / 10 = **0.005**
- Potencia estadística: ~94% para diferencias ≥ 10%
- Mínima diferencia detectable post-Bonferroni: ~14%

**Experimento ejecutado por:** `scripts/experiment_heuristics_rigorous.py`  
**Tiempo total:** 73.8 minutos | **Resultados en:** `models/mate/rigorous_*.csv`